# Bronze - Ingestão Referências
Quatro tabelas que nomeiam o que o VRA guarda como código.

In [0]:
from pyspark.sql import functions as F

REF = "/Volumes/voe_bem/bronze/arquivos/referencias/"

SEM_ASPAS = chr(0)

# Aeródromos - latin-1 e aspas que não são aspas
Duas armadilhas neste arquivo: ele não é UTF-8 e ele usa aspas (") como símbolo de segundo nas coordenadas.

In [0]:
aerodromos = (
    spark.read.format('csv')
    .option('sep', ';')
    .option('header', 'true')
    .option('skipRows', 1)
    .option('encoding', 'ISO-8859-1') # primeiro fix
    .option('quote', SEM_ASPAS) # segundo fix
    .load(f"{REF}AerodromosPublicos.csv")
)

In [0]:
import re
import unicodedata

def para_snake_case(nome):
    split_accents = unicodedata.normalize('NFKD', nome) # separa acento de palavra. ex: á -> a + ´
    sem_acento = ''.join([c for c in split_accents if not unicodedata.combining(c)]) # unicodedata.combining
    # checa se um caracter é um "combining mark" = acento que modifica letra base -> se nao, add na lista

    minusculo = sem_acento.lower()

    snake = re.sub(r'[^a-z0-9]+', '_', minusculo)
    snake = snake.strip('_')
    return snake

aerodromos = aerodromos.select(
    *[F.col(f"`{col}`").cast('string').alias(para_snake_case(col)) 
      for col in aerodromos.columns]
)

aerodromos = aerodromos.withColumn('_ingerido_em', F.current_timestamp())

In [0]:
aerodromos.write.format('delta').mode('overwrite').option(
    'overwriteSchema', 'true'
).saveAsTable('voe_bem.bronze.aerodromos')

print(f"bronze.aerodromos: {spark.table('voe_bem.bronze.aerodromos').count():,} linhas")

display(spark.sql("SELECT codigo_oaci, nome, municipio, uf FROM voe_bem.bronze.aerodromos WHERE codigo_oaci IN ('SBRB', 'SBGR', 'SBSP', 'SBFZ')"))

# Empresas - dois cadastros, duas tabelas

In [0]:
def ler_empresas(arquivo: str):
    df_bruto = (
        spark.read.format('csv')
        .option('sep', ';')
        .option('header', 'true')
        .option('skipRows', 1)
        .option('encoding', 'UTF-8')
        .option('quote', '"')
        .load(f"{REF}{arquivo}")
    )

    columns = df_bruto.columns

    df_bruto = df_bruto.select(
        *[F.col(f"`{col}`").cast('string').alias(para_snake_case(col)) 
        for col in columns]
    )

    df_bruto = df_bruto.withColumn('_ingerido_em', F.current_timestamp())

    return df_bruto

for arquivo, tabela in [
    ("pda_empresas_aereas_nacionais.csv", "voe_bem.bronze.empresas_nacionais"),
    ("pda_empresas_aereas_estrangeiros.csv", "voe_bem.bronze.empresas_estrangeiras")
]:
    df = ler_empresas(arquivo)
    df.write.format('delta').mode('overwrite').option(
        "overwriteSchema", "true"
    ).saveAsTable(tabela)

    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

In [0]:
display(spark.sql("""
    SELECT 'empresas_nacionais' AS tabela, COUNT(*) AS linhas, 
        COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM voe_bem.bronze.empresas_nacionais
    UNION ALL
    SELECT 'empresas_estrangeiras' AS tabela, COUNT(*) AS linhas, 
        COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END)
    FROM voe_bem.bronze.empresas_estrangeiras
"""))

display(spark.sql("""
    SELECT icao, razao, servico, ativa as situacao
    FROM voe_bem.bronze.empresas_estrangeiras
    WHERE icao IN ('AAL', 'TAP', 'AVA', 'ARG')
    ORDER BY icao
"""))

In [0]:
CODIGOS = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "8", "Etapa de Voo Charter"),
    ("codigo_di", "9", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

codigos = spark.createDataFrame(CODIGOS, "dominio string, codigo string, descricao string")
codigos.write.format('delta').mode('overwrite').option(
    'overwriteSchema', 'true'
).saveAsTable('voe_bem.bronze.codigos_operacao')

print(f"bronze.codigos_operacao: {spark.table('voe_bem.bronze.codigos_operacao').count():,} linhas")
display(spark.table('voe_bem.bronze.codigos_operacao'))